In [4]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =====================================================
# Load Dataset
# =====================================================

df = pd.read_csv(r"C:\Users\HP\Downloads\movies_data.csv")

# =====================================================
# Create Target Variable
# =====================================================

median_revenue = df["Revenue(INR)"].median()

df["Movie_Status"] = df["Revenue(INR)"].apply(
    lambda x: "Hit" if x >= median_revenue else "Flop"
)

# =====================================================
# Save Target
# =====================================================

y = df["Movie_Status"]

# =====================================================
# Drop Unnecessary Columns
# =====================================================

df = df.drop(columns=[
    "Movie Name",
    "Lead Star",
    "Director",
    "Music Director",
    "Movie_Status"
])

# =====================================================
# One Hot Encoding
# =====================================================

categorical_columns = [
    "Genre",
    "Release Period",
    "Whether Remake",
    "Whether Franchise",
    "New Actor",
    "New Director",
    "New Music Director"
]

X = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True
)

# =====================================================
# Train Test Split
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =====================================================
# Random Forest Classifier
# =====================================================

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

# =====================================================
# Prediction
# =====================================================

y_pred = model.predict(X_test)

# =====================================================
# Accuracy
# =====================================================

print("Accuracy :", accuracy_score(y_test, y_pred))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred))

# =====================================================
# Feature Importance
# =====================================================

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 15 Important Features\n")
print(importance.head(15))

# =====================================================
# Save Model
# =====================================================

pickle.dump(model, open("movie_rf.pkl", "wb"))
pickle.dump(X.columns.tolist(), open("rf_columns.pkl", "wb"))

print("\nModel Saved Successfully")

Accuracy : 1.0

Classification Report

              precision    recall  f1-score   support

        Flop       1.00      1.00      1.00       169
         Hit       1.00      1.00      1.00       171

    accuracy                           1.00       340
   macro avg       1.00      1.00      1.00       340
weighted avg       1.00      1.00      1.00       340


Confusion Matrix

[[169   0]
 [  0 171]]

Top 15 Important Features

                   Feature  Importance
1             Revenue(INR)    0.487513
0        Number of Screens    0.221859
2              Budget(INR)    0.209626
19           New Actor_Yes    0.018903
21  New Music Director_Yes    0.018279
20        New Director_Yes    0.018271
3              Genre_adult    0.006986
18   Whether Franchise_Yes    0.004756
7              Genre_drama    0.002914
17      Whether Remake_Yes    0.002350
5             Genre_comedy    0.001905
13          Genre_rom__com    0.001541
16   Release Period_Normal    0.001534
15          Genre_

In [6]:
from tkinter import *
from tkinter import ttk
import pandas as pd
import pickle

# ============================
# Load Model
# ============================

model = pickle.load(open("movie_rf.pkl", "rb"))
columns = pickle.load(open("rf_columns.pkl", "rb"))

# ============================
# Prediction Function
# ============================

def predict():

    sample = pd.DataFrame(0, index=[0], columns=columns)

    # Numeric Features
    sample["Budget(INR)"] = float(budget_entry.get())
    sample["Revenue(INR)"] = float(revenue_entry.get())

    # One Hot Encoding

    genre = genre_box.get()
    release = release_box.get()
    remake = remake_box.get()
    franchise = franchise_box.get()
    actor = actor_box.get()
    director = director_box.get()
    music = music_box.get()

    values = {
        "Genre": genre,
        "Release Period": release,
        "Whether Remake": remake,
        "Whether Franchise": franchise,
        "New Actor": actor,
        "New Director": director,
        "New Music Director": music
    }

    for key, value in values.items():

        col = f"{key}_{value}"

        if col in sample.columns:
            sample[col] = 1

    prediction = model.predict(sample)[0]

    result.config(
        text=f"Prediction : {prediction}",
        fg="green",
        font=("Arial",18,"bold")
    )

# ============================
# GUI
# ============================

root = Tk()

root.title("Movie Hit/Flop Prediction")

root.geometry("900x700")

root.configure(bg="#0B132B")

Label(
    root,
    text="🎬 Movie Hit / Flop Prediction",
    bg="#0B132B",
    fg="white",
    font=("Georgia",24,"bold")
).pack(pady=20)

frame = Frame(root,bg="#1C2541")
frame.pack(pady=10)

# ============================
# Budget
# ============================

Label(frame,text="Budget(INR)",bg="#1C2541",fg="white",font=("Arial",12)).grid(row=0,column=0,padx=10,pady=10)

budget_entry=Entry(frame,width=25,font=("Arial",12))
budget_entry.grid(row=0,column=1)

# ============================
# Revenue
# ============================

Label(frame,text="Revenue(INR)",bg="#1C2541",fg="white",font=("Arial",12)).grid(row=1,column=0,padx=10,pady=10)

revenue_entry=Entry(frame,width=25,font=("Arial",12))
revenue_entry.grid(row=1,column=1)

# ============================
# Genre
# ============================

Label(frame,text="Genre",bg="#1C2541",fg="white").grid(row=2,column=0)

genre_box=ttk.Combobox(frame,width=22,state="readonly",
values=["Action","Comedy","Drama","Romance","Thriller","Adventure","Horror"])

genre_box.grid(row=2,column=1)

# ============================
# Release
# ============================

Label(frame,text="Release Period",bg="#1C2541",fg="white").grid(row=3,column=0)

release_box=ttk.Combobox(frame,width=22,state="readonly",
values=["Holiday","Normal"])

release_box.grid(row=3,column=1)

# ============================
# Remake
# ============================

Label(frame,text="Whether Remake",bg="#1C2541",fg="white").grid(row=4,column=0)

remake_box=ttk.Combobox(frame,width=22,state="readonly",
values=["Yes","No"])

remake_box.grid(row=4,column=1)

# ============================
# Franchise
# ============================

Label(frame,text="Whether Franchise",bg="#1C2541",fg="white").grid(row=5,column=0)

franchise_box=ttk.Combobox(frame,width=22,state="readonly",
values=["Yes","No"])

franchise_box.grid(row=5,column=1)

# ============================
# New Actor
# ============================

Label(frame,text="New Actor",bg="#1C2541",fg="white").grid(row=6,column=0)

actor_box=ttk.Combobox(frame,width=22,state="readonly",
values=["Yes","No"])

actor_box.grid(row=6,column=1)

# ============================
# New Director
# ============================

Label(frame,text="New Director",bg="#1C2541",fg="white").grid(row=7,column=0)

director_box=ttk.Combobox(frame,width=22,state="readonly",
values=["Yes","No"])

director_box.grid(row=7,column=1)

# ============================
# Music Director
# ============================

Label(frame,text="New Music Director",bg="#1C2541",fg="white").grid(row=8,column=0)

music_box=ttk.Combobox(frame,width=22,state="readonly",
values=["Yes","No"])

music_box.grid(row=8,column=1)

# ============================
# Button
# ============================

Button(
    root,
    text="Predict",
    bg="#FF6B00",
    fg="white",
    font=("Arial",14,"bold"),
    command=predict
).pack(pady=20)

# ============================
# Result
# ============================

result=Label(
    root,
    text="Prediction",
    bg="#0B132B",
    fg="yellow",
    font=("Arial",18)
)

result.pack()

root.mainloop()